# OVERVIEW

This notebook serves as an example of how to use associated src .py files

## *IMPORTS*

Script was created in this project due to importing errors when coding run_scheduled_bot.py as its own .py file. Due to different working directories, so sys.path.append's got us nowhere. Was retrospectively added to all previous projects, alongside a change in the name of all plotting files since python automatically was choosing the first option (all previously had same name).

In [1]:
import sys
from pathlib import Path
if not globals().get("_TRADING_PORTFOLIO_PATHS_READY"):
    _HERE = (
        Path(__file__).resolve().parent
        if "__file__" in globals()
        else Path.cwd().resolve()
    )
    _PROJECTS = (
        "past_market_analysis",
        "technical_analysis",
        "options_pricing",
        "time_series_forecasting",
        "portfolio_construction",
        "ml_fundamentals",
        "ml_trading_signals",
        "risk_management",
        "trading_bot",
    )
    _REPO_ROOT = next(
        (
            directory
            for directory in (_HERE, *_HERE.parents)
            if all((directory / project).is_dir() for project in _PROJECTS)
        ),
        None,
    )
    if _REPO_ROOT is None:
        raise RuntimeError(
            "Could not locate the trading_portfolio repository root."
        )
    _SRC_PATHS = [
        str((_REPO_ROOT / project / "src").resolve())
        for project in _PROJECTS
    ]
    sys.path.extend(path for path in _SRC_PATHS if path not in sys.path)
    _TRADING_PORTFOLIO_PATHS_READY = True


In [ ]:
import pandas as pd
from alpaca.trading.requests import GetCalendarRequest

from pipeline import build_rebalance_preview, generate_trading_decisions, rebalance_summary, run_bot
from broker import get_client
from reporting import load_rebalance_history
from plotting_trading_bot import plot_rebalance_history

/Users/oscarlewis/Desktop/python/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## **Analysis**

There is no new signal uncovering or risk methods tested here, but moreso applying both of those projects into an automated trading system. Use the universe from ml_trading_signals alongside the momentum method which showed statistical significance. As per risk_management updates, there is no risk_management approaches taken as they all reduce final equity on this curve. The approach is therefore, rank universe by 63d momentum and select top 20%. Rebalance every 21 trading days buying equal weights, and allowing weights to drift between rebalances.

Start by creating the same universe, and calling the client.

In [3]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "JPM", "JNJ", "XOM", "WMT", "PG", "HD", "DIS", "NFLX", "AMD", "INTC", "CSCO", "ADBE", "CRM"]

bot_root = _REPO_ROOT / "trading_bot"
env_path = bot_root / ".env"
state_path = bot_root / "data" / "basket_state.json"
history_path = bot_root / "data" / "rebalance_history"

client = get_client(env_path=str(env_path))

At the time of writing, it is a weekend, hence the rebalance does not run accurately due to it not being a trading day. The decisions, and what they should be given a rebalance on any given day can be shown without it submitting.

In [4]:
clock = client.get_clock()
now = pd.Timestamp(clock.timestamp)

if now.tzinfo is None:
    now = now.tz_localize("America/New_York")
else:
    now = now.tz_convert("America/New_York")

calendar = client.get_calendar(filters=GetCalendarRequest(start=(now - pd.Timedelta(days=400)).date(), end=now.date()))

trading_dates = pd.DatetimeIndex([pd.Timestamp(session.date).normalize() for session in calendar])

completed_dates = []
for date, session in zip(trading_dates, calendar):
    close = pd.Timestamp(session.close)
    if close.tzinfo is None:
        close = close.tz_localize("America/New_York")
    else:
        close = close.tz_convert("America/New_York")

    if close <= now:
        completed_dates.append(date)

signal_date = completed_dates[-1]
decisions, decision_plan = generate_trading_decisions(tickers=tickers, signal_date=signal_date, trading_dates=trading_dates, lookback_days=63, top_frac=0.20, rebalance_days=21, state_path=state_path)

print(f"Signal date: {signal_date.date()}")
print(f"Reason: {decision_plan['reason']}")
if decisions is None:
    print(f"Existing basket retained: {decision_plan['current_basket']}")
else:
    display(decisions)

Signal date: 2026-09-04
Reason: strategy_changed


,ticker,momentum,signal_close,target_weight,signal_date
0,CRM,0.399866,259.230011,0.25,2026-09-04
1,MSFT,0.201528,499.700012,0.25,2026-09-04
2,JNJ,0.188243,275.230011,0.25,2026-09-04
3,JPM,0.153298,358.640015,0.25,2026-09-04


If the rebalance was done right now (not on a trading day, so not possible), then we would buy the four tickers above at equal weights, and allow the weights to deviate (there are no trades between rebalance periods), before changing the basket at the next rebalance day.

In [5]:
try:
    preview = build_rebalance_preview(client=client, tickers=tickers, env_path=env_path, state_path=state_path, lookback_days=63, top_frac=0.20, rebalance_days=21)
except ValueError as error:
    if not str(error).startswith("Invalid quote for "):
        raise

    preview = None
    print(f"Live preview unavailable while the market is closed: {error}")

if preview is not None:
    print(f"Status: {preview['status']}")
    print(f"Signal date: {preview['signal_date'].date()}")
    print(f"Execution date: {preview['execution_date'].date()}")
    print(f"Execution currently allowed: {preview['execution_allowed']}")

    display(preview["decisions"])

Live preview unavailable while the market is closed: Invalid quote for CRM


The system is run using alpaca and currently is paper trading in order to test quality of the system, and that it actually works.

In [6]:

account = client.get_account()
print(f"Cash: ${account.cash}, Portfolio value: ${account.portfolio_value}")

Cash: $69709.72, Portfolio value: $102400.76


In [7]:
if preview is None:
    print("Order preview unavailable because valid executable quotes were not available.")
else:
    print(f"Rebalance needed: {preview['plan']['rebalance_needed']}")
    print(f"Inside execution window: {preview['calendar_gate']}")
    print(f"Execution allowed: {preview['execution_allowed']}")
    print(f"Blockers: {preview['blockers'] or 'None'}")

    if preview["orders"]:
        display(pd.DataFrame(preview["orders"]))
    else:
        print("No orders required.")

Order preview unavailable because valid executable quotes were not available.


In [8]:
if preview is None:
    dry_run_result = None
    print("Dry run skipped because executable quotes are unavailable.")
else:
    dry_run_result = run_bot(tickers=tickers, submit_orders=False, env_path=env_path, state_path=state_path, lookback_days=63, top_frac=0.20, rebalance_days=21)

    print(f"Mode: {dry_run_result['mode']}")
    print(f"Status: {dry_run_result['status']}")

    if dry_run_result.get("message"):
        print(dry_run_result["message"])

    if dry_run_result.get("blockers"):
        print(f"Blockers: {dry_run_result['blockers']}")

    if dry_run_result.get("orders"):
        display(pd.DataFrame(dry_run_result["orders"]))

Dry run skipped because executable quotes are unavailable.


The aim is to make this automated, which is what caused all the issues with importing. This is done in run_bot_scheduled.py, where the file aims to 'run_bot'. Notifications are recieved when a rebalance was succesful, pending or unsuccessful, with a variety of message based on why the system met each of those conditions. The system is triggered daily in cron, at 13:31, 13:36 and 13:41, 14:31, 14:36 and 14:41 UK time, due to the time difference between UK and east coast US. Run_bot_scheduled has a setting to only run between 9:30, 9:45 east coast, with the hour difference mentioned to account for daylight savings.

The specifc cron code ran is:

31,36,41 13,14 * * 1-5 /Users/oscarlewis/Desktop/python/.venv/bin/python -B /Users/oscarlewis/Desktop/python/trading_portfolio/trading_bot/src/run_bot_scheduled.py >> /Users/oscarlewis/Desktop/python/trading_portfolio/trading_bot/data/cron.log 2>&1

The function below gives a summary of all rebalances completed.

In [9]:
rebalance_history, order_history = load_rebalance_history(history_path=history_path)

print(rebalance_summary(history_path=history_path))

if not rebalance_history.empty:
    display(rebalance_history.tail())

if not order_history.empty:
    display(order_history.tail())

No completed rebalances


Also have a plotting function which plots the portfolio value, notional value of buys/sells at each rebalance, and execution slippage. A plot will be available when rebalances occur.

In [10]:
if rebalance_history.empty:
    print("No completed rebalance history is available to plot.")
else:
    _ = plot_rebalance_history(rebalance_history, order_history)

No completed rebalance history is available to plot.


More analysis of the quality to follow when more time has passed.